# Test Inference v4
Workflow multi-agentico con **Pydantic `with_structured_output()`** + **generazione PDF sintetico**.
Modello: `google/gemma-4-31b-it`

In [19]:
import os, sys, json
os.chdir("/workspaces/hackaton-UNIBG-repo")
sys.path.insert(0, "src")
from dotenv import load_dotenv; load_dotenv()
print("API Key:", "OK" if os.getenv("OPEN_ROUTER_KEY") else "MANCA")

API Key: OK


## Modelli Pydantic

In [20]:
from typing import Any, Dict, List
from pydantic import BaseModel, Field

class FieldDefinition(BaseModel):
    field: str = Field(description="Nome del campo")
    type: str = Field(description="Tipo: string, number, date, list, object")
    sensitive: bool = Field(description="True se contiene dati personali")
    constraints: str = Field(description="Vincoli del campo")

class DocumentExtraction(BaseModel):
    schema_def: List[FieldDefinition] = Field(description="Schema del documento")
    data: Dict[str, Any] = Field(description="Dati estratti dal documento")

class ValidatedRecords(BaseModel):
    records: List[Dict[str, Any]] = Field(description="Record validati e corretti")

print("Modelli Pydantic definiti")

Modelli Pydantic definiti


## 1. OCR del PDF

In [21]:
from inferencev4 import extract_text_from_pdf

raw_text = extract_text_from_pdf("dataset/invoice-001.pdf")
print(raw_text[:1200])

2026-05-28 22:34:39.143 | INFO     | inferencev4:extract_text_from_pdf:50 - Estrazione OCR da: dataset/invoice-001.pdf


--- Page 1 ---
Invoice no: 51109338 et Om,

Date of issue: 04/13/2013 Ques

TECHSHOPY

Seller: Client:

Andrews, Kirby and Valdez Becker Ltd

58861 Gonzalez Prairie 8012 Stewart Summit Apt. 455
Lake Daniellefurt, IN 57228 North Douglas, AZ 95355

Tax Id: 945-82-2137 Tax Id: 942-80-0517

IBAN: GB75MCRL06841367619257

ITEMS
No. Description Qty UM Net price Net worth VAT [%] Gross
worth
1, CLEARANCE! Fast Dell Desktop 3,00 each 209,00 627,00 10% 689,70
Computer PC DUAL CORE
WINDOWS 10 4/8/16GB RAM
2. HP T520 Thin Client Computer 5,00 each 37,75 188,75 10% 207,63
AMD GX-212JC 1.2GHz 4GB RAM
TESTED !!READ BELOW!!
3: gaming pc desktop computer 1,00 each 400,00 400,00 10% 440,00
4. 12-Core Gaming Computer 3,00 each 464,89 1 394,67 10% 1 534,14
Desktop PC Tower Affordable
GAMING PC 8GB AMD Vega RGB
5. Custom Build Dell Optiplex 9020 5,00 each 221,99 1 109,95 10% 1 220,95
MT i5-4570 3.20GHz Desktop
Computer PC
6. Dell Optiplex 990 MT Computer 4,00 each 269,95 1 079,80 10% 1 187,78
PC Quad Core 

## 2. Agente Analizzatore

In [23]:
from inferencev4 import build_llm

llm = build_llm(0.2)
structured_llm = llm.with_structured_output(DocumentExtraction)

prompt = (
    "Analizza il seguente testo OCR estratto da un PDF.\n\n"
    "TESTO OCR:\n" + raw_text + "\n\n"
    "Il tuo compito:\n"
    "1. Deduci la struttura del documento (NON fare assunzioni sul tipo).\n"
    "2. Per ogni campo, indica nome, tipo, se e' sensibile, e eventuali vincoli.\n"
    "3. Estrai i dati dal documento nel formato dedotto.\n\n"
    "Marca come sensitive: true TUTTI i campi con dati personali."
)

extraction: DocumentExtraction = structured_llm.invoke(prompt)

schema_list = [fd.model_dump() for fd in extraction.schema_def]
print("=== SCHEMA ===")
print(json.dumps(schema_list, indent=2))
print("\n=== DATI SORGENTE ===")
print(json.dumps(extraction.data, indent=2))

=== SCHEMA ===
[
  {
    "field": "invoice_number",
    "type": "string",
    "sensitive": false,
    "constraints": "Alfanumerico"
  },
  {
    "field": "issue_date",
    "type": "date",
    "sensitive": false,
    "constraints": "Formato MM/DD/YYYY"
  },
  {
    "field": "seller_name",
    "type": "string",
    "sensitive": true,
    "constraints": "Nome azienda o persona"
  },
  {
    "field": "seller_address",
    "type": "string",
    "sensitive": true,
    "constraints": "Indirizzo completo"
  },
  {
    "field": "seller_tax_id",
    "type": "string",
    "sensitive": true,
    "constraints": "Codice fiscale/Partita IVA"
  },
  {
    "field": "client_name",
    "type": "string",
    "sensitive": true,
    "constraints": "Nome azienda o persona"
  },
  {
    "field": "client_address",
    "type": "string",
    "sensitive": true,
    "constraints": "Indirizzo completo"
  },
  {
    "field": "client_tax_id",
    "type": "string",
    "sensitive": true,
    "constraints": "Codice fis

## 3. Agente Generatore

In [24]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

N = 1
schema_json = json.dumps(schema_list, indent=2)
source_json = json.dumps(extraction.data, indent=2)

llm2 = build_llm(0.7)
generator = create_agent(
    model=llm2,
    tools=[],
    system_prompt=(
        "Sei un anonimizzatore di dati. Produci ESATTAMENTE " + str(N) + " record. "
        "I campi NON sensibili (quantita', prezzi, totali, IVA, date) "
        "DEVONO rimanere IDENTICI all'originale. "
        "Sostituisci SOLO i campi sensitive (nomi, indirizzi, tax ID, IBAN) con valori fittizi. "
        "NON alterare il numero di items. Output SOLO array JSON."
    ),
)

prompt = (
    "SCHEMA:\n" + schema_json +
    "\n\nDATI ORIGINALI (preserva TUTTI i valori non sensitive):\n" + source_json +
    "\n\nAnonimizza solo i campi sensitive."
)

result = generator.invoke({"messages": [HumanMessage(content=prompt)]})
generated_raw = result["messages"][-1].content
print("Generatore output (primi 500 char):", generated_raw[:500])

Generatore output (primi 500 char): ```json
[
  {
    "invoice_number": "51109338",
    "issue_date": "04/13/2013",
    "seller_name": "Rossi Mario Srl",
    "seller_address": "Via Roma 123, 00100 Roma, RM",
    "seller_tax_id": "IT01234567890",
    "client_name": "Bianchi Snc",
    "client_address": "Corso Vittorio Emanuele 45, 20100 Milano, MI",
    "client_tax_id": "IT98765432101",
    "iban": "IT00X0000000000000000000000",
    "items": [
      {
        "no": 1,
        "description": "CLEARANCE! Fast Dell Desktop Computer PC 


## 4. Agente Validatore

In [25]:
llm3 = build_llm(0.1)
structured_val = llm3.with_structured_output(ValidatedRecords)

prompt = (
    "Valida i seguenti record sintetici contro lo schema fornito.\n\n"
    "SCHEMA e constraints:\n" + schema_json +
    "\n\nRECORD DA VALIDARE:\n" + generated_raw +
    "\n\nPer OGNI record: verifica campi, tipi, constraints. "
    "Correggi errori di calcolo. Rimuovi record invalidi."
)

validation: ValidatedRecords = structured_val.invoke(prompt)
validated = validation.records

print(f"Validati: {len(validated)} record")
print(json.dumps(validated[0], indent=2)[:600])

Validati: 1 record
{
  "invoice_number": "51109338",
  "issue_date": "04/13/2013",
  "seller_name": "Rossi Mario Srl",
  "seller_address": "Via Roma 123, 00100 Roma, RM",
  "seller_tax_id": "IT01234567890",
  "client_name": "Bianchi Snc",
  "client_address": "Corso Vittorio Emanuele 45, 20100 Milano, MI",
  "client_tax_id": "IT98765432101",
  "iban": "IT00X0000000000000000000000",
  "items": [
    {
      "no": 1,
      "description": "CLEARANCE! Fast Dell Desktop Computer PC DUAL CORE WINDOWS 10 4/8/16GB RAM",
      "qty": 3.0,
      "um": "each",
      "net_price": 209.0,
      "net_worth": 627.0,
      "vat_p


## 5. Generazione PDF sintetico
Crea un PDF con la stessa struttura dell'originale ma con dati sintetici.

In [28]:
from fpdf import FPDF

record = validated[0]

pdf = FPDF()
pdf.add_page()
pdf.set_auto_page_break(auto=True, margin=15)

# Titolo
pdf.set_font("Helvetica", "B", 24)
pdf.set_text_color(30, 60, 120)
pdf.cell(0, 12, "INVOICE", new_x="LMARGIN", new_y="NEXT", align="C")
pdf.ln(4)

pdf.set_draw_color(30, 60, 120)
pdf.set_line_width(0.5)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
pdf.ln(6)

# Seller / Client
pdf.set_font("Helvetica", "B", 11)
pdf.set_text_color(80, 80, 80)
pdf.cell(95, 7, "SELLER", new_x="RIGHT", new_y="LAST")
pdf.cell(95, 7, "CLIENT", new_x="LMARGIN", new_y="NEXT")

pdf.set_draw_color(200, 200, 200)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
pdf.ln(3)

pdf.set_font("Helvetica", "", 10)
pdf.set_text_color(40, 40, 40)

seller_lines = []
client_lines = []
for fd in extraction.schema_def:
    name = fd.field
    val = str(record.get(name, ""))[:50]
    if not val or val == "None":
        continue
    if "seller" in name.lower() or name.lower() in ["company_name", "tax_id", "iban"]:
        seller_lines.append(val)
    elif "client" in name.lower() or "bill_to" in name.lower() or "ship_to" in name.lower():
        client_lines.append(val)

max_lines = max(len(seller_lines), len(client_lines), 1)
for i in range(max_lines):
    s = seller_lines[i] if i < len(seller_lines) else ""
    c = client_lines[i] if i < len(client_lines) else ""
    pdf.cell(95, 6, s, new_x="RIGHT", new_y="LAST")
    pdf.cell(95, 6, c, new_x="LMARGIN", new_y="NEXT")

pdf.ln(4)

# Info documento
pdf.set_font("Helvetica", "B", 10)
pdf.set_text_color(80, 80, 80)
for fd in extraction.schema_def:
    name = fd.field
    if any(kw in name.lower() for kw in ["invoice", "doc_id", "date", "number"]):
        val = str(record.get(name, ""))[:50]
        if val and val != "None":
            pdf.cell(0, 6, f"{name.replace('_', ' ').title()}: {val}", new_x="LMARGIN", new_y="NEXT")

pdf.ln(6)

# Items table
items = record.get("items", record.get("line_items", []))
if items:
    col_widths = [12, 62, 18, 18, 28, 26, 26]
    headers = ["#", "Description", "Qty", "UM", "Net Price", "Net Worth", "Gross"]

    pdf.set_fill_color(30, 60, 120)
    pdf.set_text_color(255, 255, 255)
    pdf.set_font("Helvetica", "B", 9)
    for h, w in zip(headers, col_widths):
        pdf.cell(w, 8, h, border=0, fill=True, align="C", new_x="RIGHT", new_y="LAST")
    pdf.ln()

    pdf.set_text_color(40, 40, 40)
    pdf.set_font("Helvetica", "", 9)
    fill = False
    for idx, item in enumerate(items):
        pdf.set_fill_color(245, 245, 250) if fill else pdf.set_fill_color(255, 255, 255)
        row = [
            str(idx + 1),
            str(item.get("description", item.get("name", "")))[:40],
            str(item.get("qty", item.get("quantity", "")))[:8],
            str(item.get("um", item.get("unit", "each")))[:8],
            str(item.get("net_price", item.get("unit_price", "")))[:12],
            str(item.get("net_worth", item.get("net_value", "")))[:12],
            str(item.get("gross_worth", item.get("gross_value", item.get("total", ""))))[:12],
        ]
        for val, w in zip(row, col_widths):
            pdf.cell(w, 7, val, border=0, fill=True, align="C", new_x="RIGHT", new_y="LAST")
        pdf.ln()
        fill = not fill

pdf.ln(4)

# Summary
pdf.set_font("Helvetica", "B", 10)
pdf.set_text_color(80, 80, 80)
pdf.cell(0, 7, "SUMMARY", new_x="LMARGIN", new_y="NEXT")
pdf.set_draw_color(200, 200, 200)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
pdf.ln(3)

pdf.set_font("Helvetica", "", 10)
pdf.set_text_color(40, 40, 40)
for fd in extraction.schema_def:
    name = fd.field
    if any(kw in name.lower() for kw in ["total", "vat", "tax", "net", "gross", "subtotal", "amount", "summary"]):
        raw_val = record.get(name)
        if not raw_val:
            continue
        
        # Se il campo è un dizionario annidato, lo iteriamo
        if isinstance(raw_val, dict):
            for k, v in raw_val.items():
                val_str = str(v)[:40]
                pdf.cell(0, 6, f"{k.replace('_', ' ').title()}: {val_str}", new_x="LMARGIN", new_y="NEXT")
        # Altrimenti procediamo normalmente
        else:
            val_str = str(raw_val)[:40]
            if val_str and val_str != "None":
                pdf.cell(0, 6, f"{name.replace('_', ' ').title()}: {val_str}", new_x="LMARGIN", new_y="NEXT")


# Footer
pdf.ln(10)
pdf.set_font("Helvetica", "I", 8)
pdf.set_text_color(150, 150, 150)
pdf.cell(0, 5, "This is a synthetic document generated for testing purposes.", align="C", new_x="LMARGIN", new_y="NEXT")

os.makedirs("output/notebook_v4", exist_ok=True)
pdf_path = "output/notebook_v4/synthetic_invoice.pdf"
pdf.output(pdf_path)
print(f"PDF generato: {pdf_path}")

PDF generato: output/notebook_v4/synthetic_invoice.pdf


## Bonus: workflow completo in un colpo solo

In [17]:
from inferencev4 import build_graph

app = build_graph()

result = app.invoke({
    "messages": [
        HumanMessage(content="1"),
        HumanMessage(content="run_id:notebook_v4_full"),
    ],
    "raw_text": raw_text,
    "document_schema": "",
    "source_data": "",
    "generated_data": "",
    "validated_data": "",
    "final_data": [],
    "errors": [],
    "pdf_path": "",
})

print(f"Completato: {len(result['final_data'])} record + PDF: {result['pdf_path']}")

2026-05-28 22:23:26.531 | INFO     | inferencev4:analyze_node:92 - [Agente 1] Analisi del documento - deduzione struttura e estrazione dati
2026-05-28 22:28:29.498 | INFO     | inferencev4:analyze_node:116 - Schema dedotto: 25 campi
2026-05-28 22:28:29.506 | INFO     | inferencev4:generate_node:127 - [Agente 2] Generazione di 1 record sintetici
2026-05-28 22:28:46.706 | INFO     | inferencev4:generate_node:156 - Generati 1 record sintetici
2026-05-28 22:28:46.709 | INFO     | inferencev4:validate_node:161 - [Agente 3] Validazione e correzione dei record generati
2026-05-28 22:29:13.504 | INFO     | inferencev4:validate_node:184 - Validazione completata: 1 record
2026-05-28 22:29:13.507 | INFO     | inferencev4:generate_pdf_node:221 - [Agente 5] Generazione PDF sintetico
2026-05-28 22:29:13.622 | INFO     | inferencev4:generate_pdf_node:383 - PDF sintetico salvato: output/notebook_v4_full/synthetic_invoice.pdf
2026-05-28 22:29:13.625 | INFO     | inferencev4:save_node:389 - [Salvataggio

Completato: 1 record + PDF: output/notebook_v4_full/synthetic_invoice.pdf
